> Initialize required credentials and packages (no key required)

In [1]:
%pip install numpy pandas requests seaborn matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import requests
import itertools
import json
import os

botname = 'ds6021'
version = '0.0'
purpose = 'classproject'
email = 'haz9xm@virginia.edu'

useragent = f'{botname}/{version} ({purpose}, {email}) python-requests:{requests.__version__}'
headers = {'User-Agent': useragent}

Universal Variables we need to determine

In [3]:
lineitemrequest = [
    'Assets',
    'AssetsCurrent',
    'DebtCurrent',
    'DepreciationAndAmortization',
    'GrossProfit',
    'InventoryNet',
    'LiabilitiesCurrent',
    'LongTermDebt',
    'NetIncomeLoss',
    'OperatingIncomeLoss',
    'PrepaidExpenseCurrent',
    'Revenues',
    'StockholdersEquity'
    ]
complist = ['AAPL','GOOGL','NVDA'] #small example list

> Pull list of company tickers with corresponding CIK #'s and company names. We can use this to look up CIK # and company name (title) by ticker.

In [4]:
tickurl = 'https://sec.gov/files/company_tickers.json'
rtick = requests.get(tickurl, headers = headers)
rtickjson = rtick.json()
rticklist = [rtickjson[comp] for comp in rtickjson]
ciktable = pd.json_normalize(rticklist)
#Add leading 0's to CIK number
ciktable['cik_str'] = ciktable['cik_str'].astype(str).str.zfill(10)
ciktable.head(3)

,cik_str,ticker,title
0,0001045810,NVDA,NVIDIA CORP
1,0000320193,AAPL,Apple Inc.
2,0001652044,GOOGL,Alphabet Inc.


>Filter list of companies to relevant companies

In [5]:
filteredciktable = ciktable[ciktable['ticker'].isin(complist)]

>Pull metadata (list of all filed documents with attributes) for all companies (dictionary of dictionaries form)

In [6]:
def fslist(cik,headers):    
    filingsurl = f'https://data.sec.gov/submissions/CIK{cik}.json'
    filingMetadata = requests.get(filingsurl,headers=headers)
    filingMetadatajson = filingMetadata.json()
    allFormsdict = filingMetadatajson['filings']['recent']
    return allFormsdict

allFSMeta = {cik: fslist(cik,headers) for cik in filteredciktable['cik_str']}


>Pull companyfacts data (all company concepts data for all companies)

In [7]:
def lineitemkeys(cik,headers):
    factsurl = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'
    companyfacts = requests.get(factsurl,headers=headers)
    companyfactsjson = companyfacts.json()
    fslineitemconcepts = companyfactsjson['facts']['us-gaap'] #This is a dictionary with all available line item data
    fslineitemkeys = list(fslineitemconcepts.keys()) #This is a list of all available line item keys.
    return fslineitemkeys

allFSKeys = {cik: lineitemkeys(cik,headers) for cik in filteredciktable['cik_str']}

>Pull companyconcepts data (dataframes for each line item showing values for each fs for each company for each filing date)

Determine line items to request and validate they exist in all companies' 10k's. (function should return "valid")

In [8]:
def checklineitem(lineitemrequest,allFSKeys):
    # Checking for duplicate values
    if len(set(lineitemrequest))!=len(lineitemrequest):
        return print('There are duplicate keys')
    # Convert lineitemrequest to set
    checkset = set(lineitemrequest)
    # Checking for common values in lists of allFSKeys
    alllists = [allFSKeys[cik] for cik in allFSKeys]
    check = set(alllists[0])
    for sublist in alllists[1:]:
        check.intersection_update(sublist)
        if not check:
            break
    # Checking for common values between check and lineitemrequest
    check.intersection_update(checkset)
    # Comparing check and checkset
    if check != checkset:
        notcommon = check ^ checkset
        return print(f'The following items were not common: {notcommon}')
    # All tests passed, return valid
    return print('Valid')

In [9]:
checklineitem(lineitemrequest,allFSKeys)


The following items were not common: {'PrepaidExpenseCurrent', 'GrossProfit', 'DepreciationAndAmortization', 'DebtCurrent'}


Create dictionary of df's for requested companies and line-items.

In [10]:
def lineitemdf(lineitem,cik,headers,byitem='both',):
    concepturl = f'https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{lineitem}.json'
    companyconcept = requests.get(concepturl,headers = headers)
    companyconceptjson = companyconcept.json()
    if 'USD' in list(companyconceptjson['units'].keys()):
        fslineitem = pd.DataFrame(companyconceptjson['units']['USD'])
    elif 'USD/shares' in list(companyconceptjson['units'].keys()):
        fslineitem = pd.DataFrame(companyconceptjson['units']['USD/shares'])
    else:
        return print(f'{cik}: {companyconceptjson['units'].keys()}')
    if byitem == 'cik':
        fslineitem.insert(loc=0,column = 'cik_str',value = cik)
    elif byitem == 'lineitem':
        fslineitem.insert(loc=0, column = 'line_item',value = lineitem)
    elif byitem == 'both':
        fslineitem.insert(loc=0, column = 'line_item',value = lineitem)
        fslineitem.insert(loc=0,column = 'cik_str',value = cik)
    fslineitem['filed'] = pd.to_datetime(fslineitem['filed'])
    return fslineitem

In [11]:
lineitemdfbycik = {cik: pd.concat([(lineitemdf(lineitem, cik, headers,'lineitem')) for lineitem in lineitemrequest],axis=0,ignore_index=True) for cik in filteredciktable['cik_str']}
lineitemdfbyitem = {lineitem: pd.concat([(lineitemdf(lineitem, cik, headers,'cik')) for cik in filteredciktable['cik_str']],axis=0,ignore_index=True) for lineitem in lineitemrequest}
lineitemdfbyboth = pd.concat([(lineitemdf(lineitem, cik, headers)) for cik in filteredciktable['cik_str'] for lineitem in lineitemrequest],axis=0,ignore_index=True)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

> Create dataframe with relevant metrics for all companies requested.

In [ ]:
metrics = pd.DataFrame({})

### 1. Profit